Here's the short version of what's scoped so far for Capstone 4 (Goodreads NLP/Recommendation project):

**Goal:** Build an NLP-driven recommendation project using your own reading history.

**Data sources:** (1) a CSV export of your Goodreads library, (2) some Reddit scraping — presumably book-discussion subreddits for opinions/mentions.

**Method:** an explicit NLP technique — either fine-tuned sentiment analysis or topic modeling (not yet decided which).

**Outputs:** trending books, plus preference-based recommendations.

**Status:** runs parallel to your DL roadmap, not gating it — this was the plan as last scoped, but a few real decisions were never actually pinned down.

That's genuinely all that's on record — the exact NLP method, what "trending" means operationally, and which piece to build first were left open. Given you're starting fresh today, want to nail those down before we touch any code?Here's a short outline, sequenced so the NLP method decision comes after we actually see what data looks like — matching your "not sure yet" call:

**Phase 1 — Data collection & exploration**
Export your Goodreads library (Goodreads has a built-in CSV export under Account Settings → Import/Export). Bring that CSV in and we look at what's actually there — ratings, shelves, dates, and critically, how much of your own review *text* exists, since that decides a lot downstream. Then scope the Reddit side: which subreddits (r/books, r/suggestmeabook, etc.) and what we're pulling — book mentions, genre discussion threads — using something like PRAW for the Reddit API.

**Phase 2 — NLP method decision**
Once we can see the data: if you've got substantial personal review text, sentiment analysis on your own reviews is the natural fit. If it's thinner on your own text but richer across many books/Reddit threads, topic modeling fits better for surfacing themes and "trending" signals. Prototype the chosen method on a small sample before committing.

**Phase 3 — Recommendation logic**
Build a preference profile from your library (genres/authors/themes you rate highly), a trending signal from Reddit (frequency/sentiment of book mentions over time), and combine them into a ranked recommendation output.

**Phase 4 — Wrap-up**
Same pattern as your Michelin capstone — notebook, GitHub, README.

First concrete step, since nothing downstream can really be decided without it: go grab your Goodreads CSV export and drop it in here. I don't have a way to pull it automatically (no Goodreads connector exists), so that one's on you — everything else in Phase 1 can start the moment you have it.Saved the outline so we've got it for next time too. Go grab that Goodreads export whenever you're ready, and we'll pick up Phase 1 as soon as you drop it in.

In [6]:
import pandas as pd 

In [7]:
reads = pd.read_csv('/kaggle/input/datasets/tj2555/goodreadsdata/Goodreads Library Export.csv')

reads.head()

,Book Id,Title,Author,Author l-f,Additional Authors,ISBN,ISBN13,My Rating,Publisher,Binding,...,Date Read,Date Added,Bookshelves,Bookshelves with positions,Exclusive Shelf,My Review,Spoiler,Private Notes,Read Count,Owned Copies
0,223001257,The Correspondent,Virginia Evans,"Evans, Virginia",NaN,"=""0593798430""","=""9780593798430""",0.0,Crown Publishing,Hardcover,...,NaN,2026/09/09,to-read,to-read (#165),to-read,NaN,NaN,NaN,0,0
1,238226942,Yesteryear,Caro Claire Burke,"Burke, Caro Claire",NaN,"=""""","=""""",0.0,Knopf,Hardcover,...,NaN,2026/09/09,to-read,to-read (#164),to-read,NaN,NaN,NaN,0,0
2,243596291,Fruit Fly,Josh Silver,"Silver, Josh",NaN,"=""""","=""9798892426374""",0.0,Crooked Lane Books,Paperback,...,NaN,2026/09/08,to-read,to-read (#163),to-read,NaN,NaN,NaN,0,0
3,40727626,Free Food for Millionaires,Min Jin Lee,"Lee, Min Jin",NaN,"=""""","=""""",0.0,Grand Central Publishing,Kindle Edition,...,NaN,2026/09/08,to-read,to-read (#162),to-read,NaN,NaN,NaN,0,0
4,220341389,Everything Is Tuberculosis: The History and Pe...,John Green,"Green, John",NaN,"=""0525556575""","=""9780525556572""",5.0,Crash Course Books,Hardcover,...,2026/09/08,2025/08/12,NaN,NaN,read,NaN,NaN,NaN,1,0


In [8]:
reads.columns

Index(['Book Id', 'Title', 'Author', 'Author l-f', 'Additional Authors',
       'ISBN', 'ISBN13', 'My Rating', 'Publisher', 'Binding',
       'Number of Pages', 'Year Published', 'Original Publication Year',
       'Date Read', 'Date Added', 'Bookshelves', 'Bookshelves with positions',
       'Exclusive Shelf', 'My Review', 'Spoiler', 'Private Notes',
       'Read Count', 'Owned Copies'],
      dtype='object')

- **Book Id** — Goodreads' internal unique ID for the book.
- **Title** — the book's title.
- **Author** — primary author, "First Last" format.
- **Author l-f** — same author, "Last, First" format.
- **Additional Authors** — co-authors/other contributors beyond the primary one.
- **ISBN** — 10-digit ISBN.
- **ISBN13** — 13-digit ISBN.
- **My Rating** — your personal star rating (1-5, or 0 if unrated).
- **Publisher** — publishing company.
- **Binding** — format (Hardcover, Paperback, Kindle Edition, etc.).
- **Number of Pages** — page count for that edition.
- **Year Published** — publication year of the specific edition you logged.
- **Original Publication Year** — year the book was first ever published.
- **Date Read** — date you finished it (blank if unread/unfinished).
- **Date Added** — date you added it to your Goodreads account.
- **Bookshelves** — all custom shelves you've filed the book under.
- **Bookshelves with positions** — same, but with your manual ranking/position within each shelf.
- **Exclusive Shelf** — your one primary status: read / currently-reading / to-read.
- **My Review** — your written review text, if you left one — the important one for the NLP decision.
- **Spoiler** — whether you flagged your review as containing spoilers.
- **Private Notes** — personal notes only visible to you.
- **Read Count** — how many times you've read it.
- **Owned Copies** — number of copies you own.

 - columns that can be dropped = Owned Copies, Spoiler, Year published, Binding
 - Check what the bookshleves, and bookshel;ves with positions and exclusive shelf all mean respectively

In [9]:
reads = reads.drop(columns = ['Owned Copies','Spoiler', 'Private Notes', 
                              'Year Published', 'Binding', 'ISBN', 'ISBN13', 'Author l-f'])

In [10]:
print(reads['Bookshelves'].value_counts())
reads['Bookshelves with positions'].value_counts()
reads['Exclusive Shelf'].value_counts()
reads.drop(columns = ['Bookshelves'], inplace = True )

Bookshelves
to-read              165
currently-reading      4
did-not-finish         4
Name: count, dtype: int64


In [11]:
reads.drop(columns = ['Read Count'], inplace = True )

In [12]:
reads.columns 

Index(['Book Id', 'Title', 'Author', 'Additional Authors', 'My Rating',
       'Publisher', 'Number of Pages', 'Original Publication Year',
       'Date Read', 'Date Added', 'Bookshelves with positions',
       'Exclusive Shelf', 'My Review'],
      dtype='object')

In [13]:
print(f"{(reads['Exclusive Shelf'] == 'read').sum()}, out of {len(reads)} books have been read")

134, out of 307 books have been read


In [14]:
reads.isnull().sum()

Book Id                         0
Title                           0
Author                          0
Additional Authors            199
My Rating                       0
Publisher                       2
Number of Pages                 0
Original Publication Year      10
Date Read                     261
Date Added                      0
Bookshelves with positions    134
Exclusive Shelf                 0
My Review                     295
dtype: int64

In [15]:
reads['Additional Authors']  = reads['Additional Authors'].fillna( "None")
reads['Additional Authors'].isna().sum()

np.int64(0)

In [16]:
reads['Additional Authors'].value_counts()

Additional Authors
None                                       199
Neil Smith                                   3
Neil   Smith                                 2
Richard Pevear, Larissa Volokhonsky          2
Maureen Howard                               2
                                          ... 
Phillip Gabriel, Ted Goossen, Chip Kidd      1
Olly Moss                                    1
Thomas Pynchon                               1
Russell Baker, C.M. Woodhouse                1
Jennifer Lee Carrell                         1
Name: count, Length: 99, dtype: int64

In [17]:
reads['Date Read'] = pd.to_datetime(reads['Date Read'])

In [18]:
reads['Date Read'].dtype

dtype('<M8[ns]')

In [19]:
reads.drop(columns = ['Date Read'], inplace = True)

In [20]:
reads['Book Id'].dtype

dtype('int64')

In [21]:
# Publisher                       2
# Original Publication Year      10

reads[reads['Publisher'].isna()]

publisher_fixes = {

    1772910 : 'Penguin Classics', 
    43185973 : 'Penguin Random House'
}

for k,v in publisher_fixes.items():
    reads.loc[reads['Book Id'] == k, 'Publisher'] = v 


In [22]:
reads[reads['Original Publication Year'].isna()]

year_mapping = {
    49094072: 2021,      # In Service of the Republic
    36580668: 2018,      # In Praise of Wasting Time
    41138379: 2019,      # How Finance Works
    61846327: 2019,      # War of Lanka
    6802629: 2009,       # Dan Brown's The Lost Symbol (parody)
    149205386: 1996,     # Homer's Odyssey
    54310867: 2021,      # The Lost Art of Doing Nothing
    129245861: 2023,     # New York Block by Block
    43185973: 2015,      # Summary of Homo Deus
    131002800: 1959,     # Flowers for Algernon and Other Stories
}

for k,v in year_mapping.items():
    reads.loc[reads['Book Id'] == k , 'Original Publication Year'] = v

In [23]:
reads['Original Publication Year'].isna().sum()

np.int64(0)

In [24]:
# Bookshelves with positions    134
# Exclusive Shelf                 0
# My Review                     295

reads.columns

reads[reads['Bookshelves with positions'].isna()]['Exclusive Shelf'].value_counts()

Exclusive Shelf
read    134
Name: count, dtype: int64

In [25]:
reads['Bookshelves with positions'] = reads['Bookshelves with positions'].fillna('None')

reads['Bookshelves with positions'].isna().sum()

np.int64(0)

In [26]:
reads['My Review'] = reads['My Review'].fillna('None')
reads['My Review'].isna().sum()

np.int64(0)

In [27]:
import numpy as np

In [28]:
reads['My Rating'].value_counts()
reads['My Rating']  = reads['My Rating'].replace(0, np.nan)

Now we do like Reddit scraping and get the data on how people percive these books and all that  - Pivoted to using the UCSD files for books Reddit API some new rules and stuff 

In [29]:
import os
os.listdir('/kaggle/input/datasets/tj2555/ucsdmysterybooks')

['Goodreads Mystery Thriller Crime Dataset.json']

In [30]:
with open('/kaggle/input/datasets/tj2555/ucsdmysterybooks/Goodreads Mystery Thriller Crime Dataset.json') as file:
    first_line = file.readline()
print(first_line)

{"isbn": "184737297X", "text_reviews_count": "15", "series": ["169353"], "country_code": "US", "language_code": "", "popular_shelves": [{"count": "159", "name": "to-read"}, {"count": "12", "name": "historical-fiction"}, {"count": "11", "name": "mystery"}, {"count": "10", "name": "historical-mystery"}, {"count": "7", "name": "medieval"}, {"count": "6", "name": "historical"}, {"count": "5", "name": "crime"}, {"count": "3", "name": "series"}, {"count": "3", "name": "ebook"}, {"count": "3", "name": "fiction"}, {"count": "3", "name": "crowner-john"}, {"count": "2", "name": "default"}, {"count": "2", "name": "books"}, {"count": "2", "name": "owned"}, {"count": "2", "name": "mystery-thrillers"}, {"count": "2", "name": "c"}, {"count": "2", "name": "library"}, {"count": "2", "name": "wish-list"}, {"count": "2", "name": "england"}, {"count": "2", "name": "medieval-england"}, {"count": "2", "name": "mystery-historical"}, {"count": "2", "name": "audiobook"}, {"count": "1", "name": "recs-not-intere

In [31]:
import json 

record = json.loads(first_line)
print(record['book_id'])
print(record['title'])

6066814
Crowner Royal (Crowner John Mystery, #13)


In [32]:
with open('/kaggle/input/datasets/tj2555/ucsdmysteryreviews/Goodreads Mystery Thriller Crime Reviews.json') as reviewfile:
    first_review = reviewfile.readline()
print(first_review)

{"user_id": "8842281e1d1347389f2ab93d60773d4d", "book_id": "6392944", "review_id": "5e212a62bced17b4dbe41150e5bb9037", "rating": 3, "review_text": "I haven't read a fun mystery book in a while and not sure I've ever read Poirot. Was looking for a fun read set in France while I was on holiday there and this didn't disappoint! Fast paced and good mystery. \n One that struck me was how similar Poirot is to Sherlock. They are both detectives, have a ex-military sidekick who is telling the story, and solve mysteries using their superior wit. Poirot seems like a French Sherlock. I'm curious if he was inspired by Sherlock.", "date_added": "Mon Jul 24 02:48:17 -0700 2017", "date_updated": "Sun Jul 30 09:28:03 -0700 2017", "read_at": "Tue Jul 25 00:00:00 -0700 2017", "started_at": "Mon Jul 24 00:00:00 -0700 2017", "n_votes": 6, "n_comments": 0}



In [33]:
data = []
path = '/kaggle/input/datasets/tj2555/ucsdmysterybooks/Goodreads Mystery Thriller Crime Dataset.json'

with open(path) as file:
    for line in file:
        data.append(json.loads(line))

df = pd.DataFrame(data)

In [34]:
data = []
path = '/kaggle/input/datasets/tj2555/ucsdmysteryreviews/Goodreads Mystery Thriller Crime Reviews.json'

with open(path) as file:
    for line in file:
        data.append(json.loads(line))

df2 = pd.DataFrame(data)

In [35]:
mys_books = df
mys_reviews = df2

In [36]:
mys_books.shape

(219235, 29)

In [37]:
mys_reviews.shape

(1849236, 11)

In [38]:
reads.columns

Index(['Book Id', 'Title', 'Author', 'Additional Authors', 'My Rating',
       'Publisher', 'Number of Pages', 'Original Publication Year',
       'Date Added', 'Bookshelves with positions', 'Exclusive Shelf',
       'My Review'],
      dtype='object')

In [39]:
mys_books.columns

Index(['isbn', 'text_reviews_count', 'series', 'country_code', 'language_code',
       'popular_shelves', 'asin', 'is_ebook', 'average_rating', 'kindle_asin',
       'similar_books', 'description', 'format', 'link', 'authors',
       'publisher', 'num_pages', 'publication_day', 'isbn13',
       'publication_month', 'edition_information', 'publication_year', 'url',
       'image_url', 'book_id', 'ratings_count', 'work_id', 'title',
       'title_without_series'],
      dtype='object')

In [40]:
reads['Book Id'].dtype
mys_books['book_id'].dtype

dtype('O')

In [41]:
reads['Book Id'].isin(mys_books['book_id'].astype(int)).sum()

np.int64(26)

In [42]:
reads.head()

,Book Id,Title,Author,Additional Authors,My Rating,Publisher,Number of Pages,Original Publication Year,Date Added,Bookshelves with positions,Exclusive Shelf,My Review
0,223001257,The Correspondent,Virginia Evans,None,NaN,Crown Publishing,285,2025.0,2026/09/09,to-read (#165),to-read,None
1,238226942,Yesteryear,Caro Claire Burke,None,NaN,Knopf,400,2026.0,2026/09/09,to-read (#164),to-read,None
2,243596291,Fruit Fly,Josh Silver,None,NaN,Crooked Lane Books,384,2026.0,2026/09/08,to-read (#163),to-read,None
3,40727626,Free Food for Millionaires,Min Jin Lee,None,NaN,Grand Central Publishing,577,2007.0,2026/09/08,to-read (#162),to-read,None
4,220341389,Everything Is Tuberculosis: The History and Pe...,John Green,None,5.0,Crash Course Books,198,2025.0,2025/08/12,None,read,None


In [43]:
len(reads[(reads['Book Id'].isin(mys_books['book_id'].astype(int))) & (reads['Exclusive Shelf'] == 'read') & (reads['My Review'] == 'None')])

21

In [44]:
mys_reviews.columns

Index(['user_id', 'book_id', 'review_id', 'rating', 'review_text',
       'date_added', 'date_updated', 'read_at', 'started_at', 'n_votes',
       'n_comments'],
      dtype='object')

In [45]:
mys_reviews = mys_reviews.groupby('book_id')['review_text'].apply(lambda x: ' '.join(x))

In [46]:
my_mys_books = mys_books[mys_books['book_id'].isin(reads['Book Id'].astype(str))]

In [47]:
mys_df = pd.merge(my_mys_books, mys_reviews, on = 'book_id', how = 'left')

In [48]:
mys_df.head()

,isbn,text_reviews_count,series,country_code,language_code,popular_shelves,asin,is_ebook,average_rating,kindle_asin,...,edition_information,publication_year,url,image_url,book_id,ratings_count,work_id,title,title_without_series,review_text
0,0385504225,21569,[446481],US,eng,"[{'count': '9279', 'name': 'currently-reading'...",,false,3.66,B002KQ6BT6,...,First Edition,,https://www.goodreads.com/book/show/6411961-th...,https://images.gr-assets.com/books/1358274396m...,6411961,377017,6600281,"The Lost Symbol (Robert Langdon, #3)","The Lost Symbol (Robert Langdon, #3)","** spoiler alert ** \n Critics aside, Dan Brow..."
1,0007119313,2592,[227083],US,eng,"[{'count': '6644', 'name': 'mystery'}, {'count...",,false,4.16,B000FC12Z0,...,Agatha Christie Signature Edition,2007,https://www.goodreads.com/book/show/853510.Mur...,https://images.gr-assets.com/books/1486131451m...,853510,25335,2285570,"Murder on the Orient Express (Hercule Poirot, ...","Murder on the Orient Express (Hercule Poirot, ...",I loved rereading this novel. I enjoyed the li...
2,1579126243,1433,[227084],US,eng,"[{'count': '2620', 'name': 'mystery'}, {'count...",,false,3.98,B000FC10IE,...,,2006,https://www.goodreads.com/book/show/16322.The_...,https://s.gr-assets.com/assets/nophoto/book/11...,16322,51072,626006,"The A.B.C. Murders (Hercule Poirot, #13)","The A.B.C. Murders (Hercule Poirot, #13)",A taunting series of murders that presented as...
3,1579126278,3670,[227081],US,eng,"[{'count': '37516', 'name': 'to-read'}, {'coun...",,false,4.20,B000FC12YQ,...,,2006,https://www.goodreads.com/book/show/16328.The_...,https://s.gr-assets.com/assets/nophoto/book/11...,16328,74002,1073110,"The Murder of Roger Ackroyd (Hercule Poirot, #4)","The Murder of Roger Ackroyd (Hercule Poirot, #4)","** spoiler alert ** \n *Update on October 25, ..."
4,0312997132,802,[],US,,"[{'count': '8917', 'name': 'to-read'}, {'count...",,false,4.00,B00268EVM2,...,,2004,https://www.goodreads.com/book/show/78982.Not_...,https://s.gr-assets.com/assets/nophoto/book/11...,78982,29385,1952681,"Not a Penny More, Not a Penny Less","Not a Penny More, Not a Penny Less","An elaborate plan to get money - in this case,..."


In [49]:
reads = pd.merge(reads, mys_df[['book_id', 'review_text']], left_on= reads['Book Id'].astype(int), right_on=mys_df['book_id'].astype(int), how='left')
mask = (reads['Exclusive Shelf'] == 'read') & (reads['My Review'] == 'None')
reads.loc[mask, 'My Review'] = reads.loc[mask, 'review_text']

In [50]:
reads[reads['Book Id'].isin(mys_df['book_id'].astype(int))]

,key_0,Book Id,Title,Author,Additional Authors,My Rating,Publisher,Number of Pages,Original Publication Year,Date Added,Bookshelves with positions,Exclusive Shelf,My Review,book_id,review_text
36,27833670,27833670,Dark Matter,Blake Crouch,None,4.0,Ballantine Books,342,2016.0,2026/02/07,None,read,"The book is a RCT, ceteris paribus of a scient...",27833670,"SPOILER FREE \n Hard to describe, is how I wou..."
47,29044,29044,The Secret History,Donna Tartt,None,5.0,Vintage Books,559,1992.0,2025/07/27,None,read,This is a great mystery that turns the who-don...,29044,This is a great mystery that turns the who-don...
52,2459785,2459785,"In the Woods (Dublin Murder Squad, #1)",Tana French,None,NaN,Penguin Books,448,2007.0,2026/03/07,to-read (#140),to-read,None,2459785,"DNF, I only got to 50 pages, the story felt le..."
55,15888,15888,An Instance of the Fingerpost,Iain Pears,None,NaN,Penguin Publishing Group,705,1997.0,2026/03/01,to-read (#139),to-read,None,15888,"The issue is of course, length. 707 pages. His..."
98,89724,89724,We Have Always Lived in the Castle,Shirley Jackson,Jonathan Lethem,NaN,Penguin Books,152,1962.0,2025/11/21,to-read (#103),to-read,None,89724,"I love this one; it's so terribly weird, just ..."
100,17899948,17899948,Rebecca,Daphne du Maurier,None,5.0,"Little, Brown and Company",449,1938.0,2025/01/08,None,read,Cannot stop thinking about all the mirroring w...,17899948,Cannot stop thinking about all the mirroring w...
114,853510,853510,"Murder on the Orient Express (Hercule Poirot, ...",Agatha Christie,None,NaN,HarperCollins,347,1934.0,2025/10/16,None,read,I loved rereading this novel. I enjoyed the li...,853510,I loved rereading this novel. I enjoyed the li...
149,18144124,18144124,I Am Pilgrim,Terry Hayes,None,1.0,Atria/Emily Bestler Books,612,2013.0,2024/12/20,to-read (#68),to-read,None,18144124,"This was very close to five stars for me, but ..."
228,131359,131359,"Death on the Nile (Hercule Poirot, #18)",Agatha Christie,None,2.0,Bantam Books,352,1937.0,2024/12/20,None,read,ok same as always Christie does it again. \n C...,131359,ok same as always Christie does it again. \n C...
229,16322,16322,"The A.B.C. Murders (Hercule Poirot, #13)",Agatha Christie,None,3.0,Black Dog & Leventhal Pub,207,1936.0,2024/12/20,None,read,A taunting series of murders that presented as...,16322,A taunting series of murders that presented as...


In [61]:
reads = reads.drop(columns = ['review_text'])

In [62]:
reads[reads['Exclusive Shelf']== 'read']

,Book Id,Title,Author,Additional Authors,My Rating,Publisher,Number of Pages,Original Publication Year,Date Added,Bookshelves with positions,Exclusive Shelf,My Review,book_id
4,220341389,Everything Is Tuberculosis: The History and Pe...,John Green,NaN,5.0,Crash Course Books,198,2025.0,2025/08/12,NaN,read,NaN,NaN
6,60318890,"The Winners (Beartown, #3)",Fredrik Backman,Neil Smith,3.0,Atria Books,671,2021.0,2026/07/22,NaN,read,NaN,NaN
7,23567071,Hooked: How to Build Habit-Forming Products,Nir Eyal,NaN,3.0,Penguin UK,256,2013.0,2026/06/29,NaN,read,NaN,NaN
10,92303,The Importance of Being Earnest,Oscar Wilde,NaN,5.0,Prestwick House,89,1895.0,2025/10/16,NaN,read,NaN,NaN
12,1220459,The Royal Game,Stefan Zweig,NaN,5.0,Central Books,96,1942.0,2025/10/16,NaN,read,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
302,8127,"Anne of Green Gables (Anne of Green Gables, #1)",L.M. Montgomery,Jennifer Lee Carrell,3.0,Signet,320,1908.0,2024/12/16,NaN,read,NaN,NaN
303,127441416,The Diary of a Young Girl,Anne Frank,NaN,3.0,Bantam Books,256,1947.0,2024/12/16,NaN,read,NaN,NaN
304,5,Harry Potter and the Prisoner of Azkaban (Harr...,J.K. Rowling,Mary GrandPré,4.0,Scholastic Inc.,547,1999.0,2024/12/16,NaN,read,NaN,NaN
305,25856606,The Fault in Our Stars,John Green,NaN,3.0,Dutton Books,313,2012.0,2024/12/16,NaN,read,NaN,NaN


In [52]:
reads.to_csv('/kaggle/working/reads_with_mys_reviews.csv', index=False)

In [53]:
reads[reads['Exclusive Shelf'] == 'read'].shape

(134, 15)

CHECKPOINT 2 HERE

In [66]:
import os
os.listdir('/kaggle/working/')

['.virtual_documents', 'reads_with_mys_reviews.csv']

In [71]:
reads = pd.read_csv('/kaggle/working/reads_with_mys_reviews.csv')
reads.drop(columns = ['key_0'], inplace = True)
reads.head()

reads[(reads['Exclusive Shelf'] == 'read') & (~reads['My Review'].isna())].count()

Book Id                       33
Title                         33
Author                        33
Additional Authors             7
My Rating                     32
Publisher                     33
Number of Pages               33
Original Publication Year     33
Date Added                    33
Bookshelves with positions     0
Exclusive Shelf               33
My Review                     33
book_id                       22
review_text                   22
dtype: int64

In [72]:
my_book_ids = set(reads['Book Id'].astype(str))

In [73]:
all_matched_reviews = []

with open('/kaggle/input/datasets/tj2555/ucsdgoodreadsalldata/Goodreads Reviews Dedup.json') as f:
    for line in f:
        record = json.loads(line)
        if record['book_id'] in my_book_ids:
           all_matched_reviews.append(record)

KeyboardInterrupt: 

In [ ]:
all_matched_books = []

with open('/kaggle/input/datasets/tj2555/ucsdgoodreadsalldata/Goodreads Books Dataset.json') as f:
    for line in f:
        record = json.loads(line)
        if record['book_id'] in my_book_ids:
           all_matched_books.append(record)

In [ ]:
all_books = pd.DataFrame(all_matched_books)
all_reviews = pd.DataFrame(all_matched_reviews)

In [ ]:
all_books.shape

In [ ]:
all_books.head()

In [ ]:
all_reviews.shape

In [ ]:
all_books.book_id.dtype

In [ ]:
reads['Book Id'].dtype

In [ ]:
reads['Book Id'].isin(all_books['book_id'].astype(int)).sum()

In [ ]:
len(reads[reads['Book Id'].isin(all_books['book_id'].astype(int)) & (reads['Exclusive Shelf'] == 'read')])

In [ ]:
reads[reads['Book Id'].isin(all_books['book_id'].astype(int)) & (reads['Exclusive Shelf'] == 'read')]['My Review'].unique()

In [ ]:
len(reads[reads['Book Id'].isin(all_books['book_id'].astype(int)) & (reads['Exclusive Shelf'] == 'read') & (reads['My Review'].isna() | (reads['My Review'] == 'None'))])

In [ ]:
all_reviews = all_reviews.groupby('book_id')['review_text'].apply(lambda x: ' '.join(x))

In [ ]:
my_all_books = all_books[all_books['book_id'].isin(reads['Book Id'].astype(str))]

In [ ]:
all_df = pd.merge(my_all_books, all_reviews, on = 'book_id', how = 'left')

In [ ]:
reads['review_text'] = reads['review_text_x'].combine_first(reads['review_text_y'])
reads = reads.drop(columns=['book_id_x', 'book_id_y', 'review_text_x', 'review_text_y'])

In [ ]:
reads.head()
reads.drop(columns = ['key_0'])

In [ ]:
mask = reads['Exclusive Shelf'] == 'read'
reads.loc[mask, ['My Review', 'review_text']].head(20)

In [ ]:
mask = (reads['Exclusive Shelf'] == 'read') & (reads['My Review'].isna() | (reads['My Review'] == 'None'))
reads.loc[mask, 'My Review'] = reads.loc[mask, 'review_text']

In [ ]:
len(reads[mask])

In [ ]:
mask = reads['Exclusive Shelf'] == 'read'
reads.loc[mask, ['My Review', 'review_text']].head(20)

In [ ]:
reads[reads['Book Id'].isin(all_df['book_id'].astype(int))]

In [ ]:
reads.drop(columns = ['review_text', 'key_0'])

In [ ]:
import os
os.listdir('/kaggle/working/')

In [ ]:
reads[reads['Exclusive Shelf'] == 'read'].shape

In [ ]:
import os
os.listdir('/kaggle/working/')

Checkpoint 3 - completely filled up the missing reviews in my Read list 

In [74]:
import pandas as pd

In [75]:
reads = pd.read_csv('/kaggle/input/datasets/tj2555/reads-checkpoint3/Reads with All Reviews.csv')
reads.head()

,key_0,Book Id,Title,Author,Additional Authors,My Rating,Publisher,Number of Pages,Original Publication Year,Date Added,Bookshelves with positions,Exclusive Shelf,My Review,review_text
0,223001257,223001257,The Correspondent,Virginia Evans,NaN,NaN,Crown Publishing,285,2025.0,2026/09/09,to-read (#165),to-read,NaN,NaN
1,238226942,238226942,Yesteryear,Caro Claire Burke,NaN,NaN,Knopf,400,2026.0,2026/09/09,to-read (#164),to-read,NaN,NaN
2,243596291,243596291,Fruit Fly,Josh Silver,NaN,NaN,Crooked Lane Books,384,2026.0,2026/09/08,to-read (#163),to-read,NaN,NaN
3,40727626,40727626,Free Food for Millionaires,Min Jin Lee,NaN,NaN,Grand Central Publishing,577,2007.0,2026/09/08,to-read (#162),to-read,NaN,NaN
4,220341389,220341389,Everything Is Tuberculosis: The History and Pe...,John Green,NaN,5.0,Crash Course Books,198,2025.0,2025/08/12,NaN,read,NaN,NaN


We now move on to do Topic Modelling: (a.) Text Cleaning and Preprocessing - here we make sure that all the punctuations, wierd stray cahrecters are removed, remove stopwords ("the", "and", "a"...), possibly lemmatize (reduce words to their root form). (b.) Vectorise - Turn each book review into like numbers that the model can run. It weighs words by how distinctive they are to a specific book's reviews, not just how often they appear overall. (c.)Run LDA (Latent Dirichlet Allocation) — the standard topic-modeling algorithm. It looks across all your books' review text and finds clusters of words that tend to co-occur, and calls each cluster a "topic." You pick how many topics to look for upfront (a number you choose, then can tune).(d) Assign each book a topic mixture — every book gets a score for how much its reviews reflect each discovered topic, not just one single label.

In [80]:
reads['My Review'] = reads['My Review'].str.lower()

reads[(reads['Exclusive Shelf'] == 'read') & (~reads['My Review'].isna())].count()
reads[reads['Exclusive Shelf'] == 'read'].count()

reads = reads.drop(columns = ['key_0','review_text'])

In [84]:
reads[~reads['My Review'].isna()].count()

Book Id                       96
Title                         96
Author                        96
Additional Authors            30
My Rating                     87
Publisher                     96
Number of Pages               96
Original Publication Year     96
Date Added                    96
Bookshelves with positions     0
Exclusive Shelf               96
My Review                     96
dtype: int64

In [85]:
reads['My Review'] = reads['My Review'].str.replace(r'[^\w\s]', '', regex = False)

In [89]:
# we want to remove stopwords like - and, the , an , a , 
# we use the NLTK - Natural Lanuage Toolkit to do a better job at this and remove them to prevent then from hoggin the resuls for us

import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [92]:
reads['My Review'] = reads['My Review'].apply(
    lambda x: ' '.join(word for word in x.split() if word not in stop_words) if isinstance(x, str) else x
)

Numbers — reviews might have stray digits (page numbers mentioned, "5 stars," ratings text). Worth stripping these too, similar regex approach (\d matches digits).
Lemmatization — reducing words to their root form (e.g., "running," "runs," "ran" all become "run"), so the model doesn't treat variations of the same word as unrelated. This uses NLTK's WordNetLemmatizer, another new tool.

In [93]:
reads['My Review'] = reads['My Review'].str.replace(r'\d' , '', regex = True)

In [95]:
# we now do lemmatazation for the My Review 

nltk.download('wordnet')
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [96]:
reads['My Review'] = reads['My Review'].apply(
    lambda x: ' '.join(lemmatizer.lemmatize(word) for word in x.split()) if isinstance(x, str) else x
)

In [97]:
reads['My Review'] = reads['My Review'].str.strip().str.replace(r'\s+', ' ', regex=True)

In [98]:
reads.to_csv('/kaggle/working/reads_checkpoint4_cleaned.csv', index=False)

import os
os.listdir('/kaggle/working/')

['.virtual_documents',
 'reads_with_mys_reviews.csv',
 'reads_checkpoint4_cleaned.csv']